# PEAK M-ATH — Suspicious Browser Extensions

Detect browser or VS Code extensions that may collect and exfiltrate sensitive data based on permissions and metadata, following the **PEAK** framework: **Prepare → Execute → Act → Knowledge**.

**Ref:** M20 — Suspicious browser extensions to collect and exfiltrate sensitive data  
**M-ATH approach:** Rule-based scoring of extension permissions and description keywords to surface high-risk add-ons.

## How to use
1. Put extension metadata CSV into `input/` (columns: name, permissions, description, etc.)
2. Run all cells
3. Review flagged extensions in `output/`

In [ ]:
pass  # Placeholder (removed environment-specific output)

## PREPARE — Plan your Approach

- **Select Topic:** Suspicious browser/VS Code extensions that abuse permissions to collect or exfiltrate data (MITRE ATT&CK [T1176](https://attack.mitre.org/techniques/T1176/) Browser Extensions).
- **Research Topic:** High-risk permission patterns, suspicious description keywords, known malicious extension campaigns.
- **Identify Datasets:** Extension metadata exports (name, permissions, description) from browser management or EDR telemetry.
- **Select Algorithms:** Rule-based scoring — high-risk permissions and suspicious keywords each contribute to a cumulative risk score.

In [ ]:
# Scenario mode: run from scenarios/suspicious_browser_extensions_to_collect_and_exfiltrate_sensitive_data/
import os
import sys
from pathlib import Path
SCENARIO_DIR = Path('.').resolve()
REPO_ROOT = SCENARIO_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
INPUT_DIR = SCENARIO_DIR / 'input'
OUTPUT_DIR = SCENARIO_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SCENARIO_MODE = True

In [ ]:
import glob
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 150)

if not globals().get('SCENARIO_MODE', False):
    INPUT_DIR = Path('input')
    OUTPUT_DIR = Path('output')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _rel(p):
    try:
        if globals().get('SCENARIO_MODE', False) and 'REPO_ROOT' in globals() and hasattr(p, 'is_relative_to') and p.is_relative_to(REPO_ROOT):
            return p.relative_to(REPO_ROOT)
        return p
    except (ValueError, AttributeError):
        return p

print(f'Input folder: {_rel(INPUT_DIR)}')
print(f'Output folder: {_rel(OUTPUT_DIR)}')

## Risk scoring for permissions and description

In [ ]:
HIGH_RISK_PERMISSIONS = [
    'clipboard', 'clipboardRead', 'clipboardWrite', 'storage', 'unlimitedStorage',
    'cookies', 'webRequest', 'webRequestBlocking', 'tabs', 'history',
    'bookmarks', 'downloads', 'management', 'debugger', 'proxy',
    'nativeMessaging', 'identity', 'privacy', 'sessions'
]

SUSPICIOUS_KEYWORDS = [
    'password', 'credential', 'login', 'wallet', 'crypto', 'bitcoin',
    'exfiltrat', 'collect', 'steal', 'keylog', 'screen capture',
    'form data', 'autofill', 'credit card', 'bank', 'inject'
]

def score_extension(row: pd.Series) -> tuple[int, list[str]]:
    """Score extension by permissions and description. Returns (score, reasons)."""
    score = 0
    reasons = []
    perms = str(row.get('permissions', '') or row.get('permission', '') or '').lower()
    desc = str(row.get('description', '') or row.get('desc', '') or '').lower()
    name = str(row.get('name', '') or row.get('id', '') or '').lower()
    combined = perms + ' ' + desc + ' ' + name
    for p in HIGH_RISK_PERMISSIONS:
        if p in perms or p in combined:
            score += 1
            reasons.append(f'perm:{p}')
    for k in SUSPICIOUS_KEYWORDS:
        if k in combined:
            score += 1
            reasons.append(f'keyword:{k}')
    if len(perms) > 200:
        score += 1
        reasons.append('many-permissions')
    return score, reasons

## EXECUTE — Experimentation Time

- **Gather Data:** Load extension metadata CSVs from `input/`.
- **Pre-Process Data:** Concatenate sources, detect relevant columns.
- **Apply:** Score each extension against high-risk permissions and suspicious keywords.
- **Analyze:** Review flagged extensions sorted by risk score.
- **Escalate Critical Findings:** Extensions with high scores and broad permissions warrant immediate review and potential removal.

## Load extension data

In [ ]:
csv_paths = sorted(glob.glob(str(INPUT_DIR / '**' / '*.csv'), recursive=True))
print(f'Found {len(csv_paths)} CSV file(s).')

if not csv_paths:
    raise FileNotFoundError(f'No CSV files in {_rel(INPUT_DIR)}. Add extension metadata (name, permissions, description).')

dfs = []
for p in csv_paths:
    df = pd.read_csv(p)
    try:
        src_rel = str(Path(p).relative_to(REPO_ROOT)) if 'REPO_ROOT' in globals() and Path(p).is_relative_to(REPO_ROOT) else p
    except (ValueError, AttributeError):
        src_rel = p
    df['_source_file'] = src_rel
    dfs.append(df)
raw = pd.concat(dfs, ignore_index=True)

print(f'Loaded {len(raw):,} extensions.')

## Score and flag suspicious extensions

In [ ]:
scored = []
for idx, row in raw.iterrows():
    score, reasons = score_extension(row)
    scored.append({
        'row_index': idx,
        'name': row.get('name', row.get('id', '')),
        'score': score,
        'reasons': ','.join(reasons) if reasons else '',
        'permissions_preview': str(row.get('permissions', ''))[:200],
        'description_preview': str(row.get('description', ''))[:200]
    })

scored_df = pd.DataFrame(scored)
flagged = scored_df[scored_df['score'] > 0].sort_values('score', ascending=False).reset_index(drop=True)

print(f'Flagged {len(flagged):,} extensions with risk score > 0.')
if len(flagged) > 0:
    display(flagged.head(20))

## ACT — Wrapping Up the Investigation

- **Document Findings:** Scored extensions exported for review and ticketing.
- **Preserve Hunt:** Results archived to `output/browser_extensions_results.csv`.
- **Create Detections / Playbooks:** High-scoring extensions can be added to blocklists or trigger automated policy enforcement.

In [ ]:
out_path = OUTPUT_DIR / 'browser_extensions_results.csv'
if len(flagged) > 0:
    flagged.to_csv(out_path, index=False)
else:
    pd.DataFrame(columns=['row_index', 'name', 'score', 'reasons', 'permissions_preview', 'description_preview']).to_csv(out_path, index=False)
print(f'Saved to {_rel(out_path)}')

## KNOWLEDGE — Continuous Improvement

- **Re-Add Topics to Backlog:** New permission patterns or extension campaigns discovered during analysis become future hunt topics.
- **Communicate Findings:** Share flagged extensions with IT administration and SOC leadership for policy updates.
- **Feed Improvements Back:** Update `HIGH_RISK_PERMISSIONS` and `SUSPICIOUS_KEYWORDS` based on emerging extension abuse patterns.
- **Measure Effectiveness:** Track flagged counts and confirmed malicious extensions across runs to assess detection quality.